<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-05-llm-api-engineering/lesson-5.3-cost-engineering/notebooks/exercises-5_3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 5.3: Cost Engineering in Python

*Module 5 · Building Production Applications*

> An unoptimized GenAI app at 100K calls/day costs ₹5 lakh/month. Cost-engineered: ₹30K. This lesson teaches four techniques that cut costs 80-95%: token counting, cheap-first-escalate routing, semantic caching, and prompt compression.

`Token Counting` · `Model Routing` · `Semantic Cache` · `Compression` · `60 min`

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-5.3.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Token Counting — Know What You Spend — `01_token_counter.py`
2. Step 2 — Cheap-First-Escalate — The 90/10 Strategy — `02_escalation.py`
3. Step 3 — Semantic Caching — Never Pay Twice — `03_semantic_cache.py`
4. Step 4 — Prompt Compression — Fewer Tokens, Same Info — `04_compression.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai numpy tiktoken


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Token Counting — Know What You Spend

You can't optimize what you don't measure.

**`01_token_counter.py`** · _Block 1 of 4_

TOKEN COUNTING & COST TRACKING


In [ ]:
# TOKEN COUNTING & COST TRACKING
import tiktoken
class CostTracker:
    PRICING = {"gemini-flash":(0.10,0.40),"gpt-4o-mini":(0.15,0.60),"gpt-4o":(2.50,10.0),"claude-sonnet":(3.0,15.0)}
    def __init__(self):
        self.enc=tiktoken.get_encoding("cl100k_base"); self.total_cost=0; self.calls=[]
    def count(self,t): return len(self.enc.encode(t))
    def log(self,model,prompt,resp):
        ti,to=self.count(prompt),self.count(resp)
        p=self.PRICING.get(model,(1,5)); c=(ti*p[0]+to*p[1])/1e6
        self.total_cost+=c; self.calls.append({"model":model,"cost":c}); return c
    def report(self):
        print(f"  {len(self.calls)} calls | Total: ${self.total_cost:.6f}")
        print(f"  Daily (10K): ${self.total_cost/max(len(self.calls),1)*10000:.2f}")
        print(f"  Monthly:     ${self.total_cost/max(len(self.calls),1)*300000:.2f}")

t=CostTracker()
t.log("gemini-flash","What is Python?","Python is a language."*5)
t.log("gpt-4o","Explain microservices..."*10,"Microservices..."*50)
t.log("claude-sonnet","Write a plan..."*20,"Executive summary..."*100)
t.report()


> **What just happened?** CostTracker counts every token, calculates cost per call, projects daily/monthly. A GPT-4o call costs 25x more than the same call on Gemini Flash. Rule: measure before optimizing.


### Step 2 · Cheap-First-Escalate — The 90/10 Strategy

Try cheapest model first. Escalate only if quality is insufficient.

**`02_escalation.py`** · _Block 2 of 4_

CHEAP-FIRST-ESCALATE ROUTING


In [ ]:
# CHEAP-FIRST-ESCALATE ROUTING
import google.generativeai as genai, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class CheapFirstRouter:
    def __init__(self):
        self.cheap=genai.GenerativeModel("gemini-2.0-flash")
        self.judge=genai.GenerativeModel("gemini-2.0-flash")
        self.stats={"cheap":0,"escalated":0}
    def _check(self,q,r):
        c=self.judge.generate_content(f"Rate 1-10: does this answer the query?\nQ:{q[:200]}\nA:{r[:300]}\nReturn ONLY the number.")
        try: return int(c.text.strip().split()[0])>=7
        except: return True
    def generate(self,query):
        r=self.cheap.generate_content(query).text.strip()
        if self._check(query,r):
            self.stats["cheap"]+=1; return r,"flash","✓ cheap"
        e=self.cheap.generate_content(f"Provide a better answer.\nQuery:{query}\nPrevious:{r}\nBetter:")
        self.stats["escalated"]+=1; return e.text.strip(),"escalated","⬆ escalated"

router=CheapFirstRouter()
for q in ["What is 2+2?","Translate hello to Hindi","Design a distributed system for 10M users"]:
    ans,model,status=router.generate(q)
    print(f"  {status} '{q[:45]}' \u2192 {ans[:60]}...")
print(f"\n  {router.stats['cheap']}/{sum(router.stats.values())} served cheap")


> **What just happened?** Simple queries passed the quality check and were served by Flash. Complex queries got escalated. 70-90% of queries pass the cheap model. Cost savings: 60-85%.


### Step 3 · Semantic Caching — Never Pay Twice

Similar question asked before? Return cached answer. Zero LLM cost.

**`03_semantic_cache.py`** · _Block 3 of 4_

SEMANTIC CACHING — Match by meaning, not exact text


In [ ]:
# SEMANTIC CACHING — Match by meaning, not exact text
import google.generativeai as genai, numpy as np, time, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class SemanticCache:
    def __init__(self,threshold=0.92):
        self.threshold=threshold; self.cache=[]; self.stats={"hits":0,"misses":0}
        self.model=genai.GenerativeModel("gemini-2.0-flash")
    def _embed(self,t):
        return np.array(genai.embed_content(model="models/text-embedding-004",content=t)["embedding"])
    def query(self,prompt):
        qe=self._embed(prompt)
        for e in self.cache:
            sim=np.dot(qe,e["emb"])/(np.linalg.norm(qe)*np.linalg.norm(e["emb"]))
            if sim>=self.threshold:
                self.stats["hits"]+=1; return e["resp"],True
        resp=self.model.generate_content(prompt).text.strip()
        self.cache.append({"emb":qe,"resp":resp})
        self.stats["misses"]+=1; return resp,False

c=SemanticCache(threshold=0.90)
for q in ["What is machine learning?","What is ML?","Explain machine learning",
         "What is deep learning?","Tell me about deep learning","How does Python work?"]:
    ans,hit=c.query(q)
    print(f"  {'⚡ HIT (free!)' if hit else '🔄 MISS (LLM)'} '{q}'")
h,m=c.stats["hits"],c.stats["misses"]
print(f"\n  Hit rate: {h}/{h+m} ({h/(h+m)*100:.0f}%) \u2192 {h} free calls")


> **What just happened?** "What is ML?" matched "What is machine learning?" (92% similarity). Cache hit = zero cost + instant. Production hit rates: 40-70%. A 60% hit rate = 60% cost AND latency reduction.


### Step 4 · Prompt Compression — Fewer Tokens, Same Info

Remove filler, abbreviate, restructure. Same answer, 65% fewer tokens.

**`04_compression.py`** · _Block 4 of 4_

PROMPT COMPRESSION — Same info, fewer tokens


In [ ]:
# PROMPT COMPRESSION — Same info, fewer tokens
import tiktoken, re
enc=tiktoken.get_encoding("cl100k_base")
def ct(t): return len(enc.encode(t))

bloated="""You are a very helpful and friendly customer support assistant for our 
company. Our company is called Netsetos and we are an educational technology company 
based in the beautiful city of Hyderabad, India. We specialize in providing high-quality 
courses in the field of Artificial Intelligence and Machine Learning. When a customer 
asks you a question, please provide a helpful, detailed, and accurate response. Please 
note that our refund policy allows full refunds within 7 days of purchase. After 7 days, 
we offer a 50% refund up to 30 days. After 30 days, no refunds are available.
Customer question: What is your refund policy?"""

compressed="""Role: Netsetos support (edtech, Hyderabad). Scope: AI/ML courses.
Refund: 7d=100%, 8-30d=50%, 30d+=none.
Q: What is your refund policy?"""

print(f"  Original:   {ct(bloated)} tokens")
print(f"  Compressed: {ct(compressed)} tokens")
print(f"  Reduction:  {(1-ct(compressed)/ct(bloated))*100:.0f}%")
print(f"  Same answer. {(1-ct(compressed)/ct(bloated))*100:.0f}% cheaper per call.")
print(f"  At 100K calls/day on GPT-4o: saves ~${(ct(bloated)-ct(compressed))*100000*2.5/1e6:.0f}/day")


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
